In [6]:
import sys
import os
sys.path.append("../../src/")
#Imports
import single_particle_sector as sps
import kblock_ising_model as kb
import numpy as np
import scipy as sp
import numpy.linalg as la
import matplotlib.pyplot as plt
from matplotlib.pyplot import plot, scatter
from collections import Counter
from tqdm import tqdm
import numpy as np
from scipy.integrate import solve_ivp
from joblib import Parallel, delayed

In [7]:
def g(t, tau, g0, gf):
    return max(g0 + (gf - g0) * t / tau, 0)

def rhs(t, v, k, tau, g0, gf):
    return [
        2 * ((g(t, tau, g0, gf) - np.cos(k)) * v[0] + np.sin(k) * v[1]) / 1j,
        2 * (-(g(t, tau, g0, gf) - np.cos(k)) * v[1] + np.sin(k) * v[0]) / 1j
    ]

def evolve_mode_all_times(i, v_0, k, t_span, t_eval, tau, g0, gf):
    v0 = [v_0[0][i] + 0j, v_0[1][i] + 0j]
    ki = k[i]
    res = solve_ivp(rhs, t_span, v0, args=(ki, tau, g0, gf), t_eval=t_eval)
    return res.y  # shape: (2, len(t_eval))



In [8]:
L = 100
g0 = 100
gf = 0
k = kb.k_vals(L)
v_0 = kb.U(k, g0)
taus = np.logspace(-4, 2, 50)
t_eval = np.linspace(0, 2 * taus[-1], 100)  # fixed time grid


In [9]:
# Storage array: (tau_index, time_index, 2, L)
x = np.empty((len(taus), 2, L), dtype=complex)

for j, tau in enumerate(tqdm(taus)):
    t_span = (0, 1.25 * tau)
    t_eval = np.linspace(*t_span, 100)  # time points to integrate over

    results = Parallel(n_jobs=-1)(
        delayed(evolve_mode_all_times)(i, v_0, k, t_span, t_eval, tau, g0, gf)
        for i in range(L)
    )

    results = np.array(results)  # shape: (L, 2, len(t_eval))
    final_uv = results[:, :, -1].T  # shape: (2, L)
    x[j] = final_uv



  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:14<00:00,  3.57it/s]


In [20]:
x[0]-v_0

array([[-4.51285238e-05-9.54768145e-03j, -4.51193008e-05-9.54670416e-03j,
        -4.51008919e-05-9.54475335e-03j, -4.50733710e-05-9.54183650e-03j,
        -4.50368485e-05-9.53796483e-03j, -4.49914710e-05-9.53315323e-03j,
        -4.49374205e-05-9.52742020e-03j, -4.48749137e-05-9.52078781e-03j,
        -4.48042014e-05-9.51328161e-03j, -4.47255667e-05-9.50493054e-03j,
        -4.46393245e-05-9.49576679e-03j, -4.45458201e-05-9.48582577e-03j,
        -4.44454274e-05-9.47514589e-03j, -4.43385477e-05-9.46376847e-03j,
        -4.42256078e-05-9.45173759e-03j, -4.41070585e-05-9.43909991e-03j,
        -4.39833724e-05-9.42590451e-03j, -4.38550424e-05-9.41220272e-03j,
        -4.37225791e-05-9.39804789e-03j, -4.35865094e-05-9.38349523e-03j,
        -4.34473738e-05-9.36860159e-03j, -4.33057244e-05-9.35342524e-03j,
        -4.31621227e-05-9.33802566e-03j, -4.30171375e-05-9.32246329e-03j,
        -4.28713422e-05-9.30679934e-03j, -4.27253129e-05-9.29109548e-03j,
        -4.25796261e-05-9.27541368e-03